# Main running
This jupyter notebook allow to choose the mode of running.
Either in one-shot or with sensivity analyses AND with shap or not.

### Imports

In [ ]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# Standard library imports
from datetime import datetime

# Local imports
from sensitivity_analysis import run_one_shot, run_sensitivity, run_per_model_configs
from constants import Constant as C

### One-shot

In [ ]:
# One-shot, default config, with SHAP
result = run_one_shot()

# Access results
result["df_report"]
result["fitted_models"]
result["shap_dict"]

In [ ]:
# One-shot, faster (no SHAP)
result = run_one_shot(run_shap=False)

# Access results
result["df_report"]
result["fitted_models"]

In [ ]:
# One-shot, override specific params only
result = run_one_shot(cfg={
    "smote_strategy"   : 0.5,
    "threshold"        : 0.4,
    "xgb_learning_rate": 0.1,
})

# Access results
result["df_report"]
result["fitted_models"]
result["shap_dict"]

### Sensitivity analyses without SHAP
Used to identify the best models configuration. 
Once done, I perfomed a sensitivity analyses with SHAP with a smaller grid focusing on models performance.

Feasible parameters for the grid:

    MODEL_KEYS = {
        "RandomForest" : {"rf_n_estimators", "rf_max_depth", "rf_min_samples_leaf",
                          "smote_strategy", "threshold", "cut_year", "apply_smote"},
        "XGBoost"      : {"xgb_n_estimators", "xgb_max_depth", "xgb_learning_rate",
                          "xgb_subsample", "smote_strategy", "threshold",
                          "cut_year", "apply_smote"},
        "AdaBoost"     : {"ada_n_estimators", "ada_learning_rate",
                          "smote_strategy", "threshold", "cut_year", "apply_smote"},
        "RUSBoost"     : {"rus_n_estimators", "rus_learning_rate",
                          "smote_strategy", "threshold", "cut_year", "apply_smote"},
        "Logit"        : {"logit_C", "logit_penalty", "smote_strategy", "threshold", "cut_year", "apply_smote"},
        "SVM"          : {"svm_C", "svm_kernel", "svm_gamma", ...},
        "MLP"          : {"mlp_hidden_layer_sizes", "mlp_alpha", "mlp_learning_rate_init", "mlp_activation", ...},
    }
    
`logit_penalty` accepts 'l2', 'l1', 'none'.

Both **Logit** and **SVM** carry their own preprocessing (sentinel -1 -> median impute,
standardisation of continuous features, sin/cos for cyclical hour/month) because they are
distance/margin based. The tree models need none of this.

Interpretation differs by model family:
* Tree models -> SHAP (beeswarm, dependence, waterfall, permutation importance)
* Logit       -> odds-ratio analysis (coefficients) + interpretation .txt reports
* SVM         -> permutation importance (any kernel); plus signed weights when kernel='linear'.
  SVM has no odds ratios (a margin weight is not a log-odds) and no SHAP TreeExplainer.

In [ ]:
# Sensitivity grid, custom axes without SHAP (faster)
df_results = run_sensitivity(
    grid={
    "smote_strategy"        : [0.2, 0.3, 0.4],
    "threshold"             : [0.3, 0.5, 0.7],
    "xgb_n_estimators"      : [200, 300],
    "xgb_learning_rate"     : [0.1, 0.2, 0.3],
    "xgb_max_depth"         : [7, 10, 13],
    "xgb_subsample"         : [0.8, 1.0],
    "rf_n_estimators"       : [100, 200, 300, 400],
    "rf_max_depth"          : [7, 10, 13],
    "ada_n_estimators"      : [200, 300],
    "ada_learning_rate"     : [0.1, 0.2, 0.3],
    "rus_n_estimators"      : [200, 300], 
    "rus_learning_rate"     : [0.1, 0.2, 0.3],
    "logit_C"               : [0.01, 0.1],
    "logit_penalty"         : ["l1", "l2"],
    "svm_C"                 : [0.5, 1.0],
    "svm_kernel"            : ["rbf", "linear"],
    "mlp_hidden_layer_sizes": [(32,), (64, 32), (128, 64, 32)],
    "mlp_alpha"             : [1e-4, 1e-3],
    "mlp_learning_rate_init": [1e-3, 1e-2],
}
)

# Find and save the best config per model across the grid
best = df_results.sort_values("f1", ascending=False).groupby("model").first()
ts = datetime.today().strftime("%Y%m%d_%Hh%M")
best.to_csv(C.DATA_PATH_OUTPUTS_EVL_REP / f"best_configs_{ts}.csv")

### Sensitivity analyses with SHAP

In [ ]:
# Sensitivity grid WITH SHAP (small grid only because it takes much longer)
df_results = run_sensitivity(
    grid={"smote_strategy": [0.3],
          "threshold": [0.3, 0.5],
          "xgb_n_estimators": [100],
          "xgb_learning_rate": [0.1],
          "rf_n_estimators": [200],
          "rf_max_depth"   : [10],
          "ada_n_estimators": [100],
          "rus_n_estimators": [100], 
          "rus_learning_rate": [0.1],
          "logit_C" : [1.0],
          "logit_penalty" : ["l2"],
          "svm_C" : [1.0],
          "svm_kernel" : ["rbf"],
          "mlp_hidden_layer_sizes": [(64, 32)],
          "mlp_alpha"             : [1e-3],
          "mlp_learning_rate_init": [1e-3],
          },
    run_shap=True,
)

# Find and save the best config per model across the grid
best = df_results.sort_values("f1", ascending=False).groupby("model").first()
ts = datetime.today().strftime("%Y%m%d_%Hh%M")
best.to_csv(C.DATA_PATH_OUTPUTS_EVL_REP / f"best_configs_{ts}.csv")

### Per-model specific configurations with SHAP analysis
Instead of sweeping the all grid, **each model can receive its own single configuration** as a dict of dicts `{model_name: {param: value}}`. Every model is trained **once** with its config and its full interpretation is produced:
* tree models (RandomForest, XGBoost, AdaBoost, RUSBoost) => SHAP,
* Logit => odds ratios, SVM => permutation importance + linear weights,
* MLP => permutation importance + directional sensitivity probe.

Each model writes to `<parent_dir>/<model_name>/` with its interpretation in
`<parent_dir>/<model_name>/shap/`. `parent_dir` defaults to `data/outputs/figures/sensitivity/best_configs/`; override it to keep a test run in a separate folder. Params not listed fall back to the one-shot defaults.

In [ ]:
# Run each model once with its own explicit configuration, with SHAP / interpretation.
specific_models = run_per_model_configs(
    model_configs={
        "RandomForest": {"apply_smote": True,
                         "smote_strategy": 0.3,
                         "threshold": 0.5,
                         "rf_n_estimators": 200, 
                         "rf_max_depth": 10,
                         "rf_max_depth": 10,
                         "rf_min_samples_leaf": 2
                         },
        "XGBoost"     : {"apply_smote": True,
                         "smote_strategy": 0.4,
                         "threshold": 0.3,
                         "xgb_n_estimators": 300, 
                         "xgb_learning_rate": 0.1,
                         "xgb_max_depth": 10,
                         "xgb_learning_rate": 0.1,
                         "xgb_subsample": 0.8,
                         },
        "AdaBoost"    : {"apply_smote": True,
                         "smote_strategy": 0.4,
                         "threshold": 0.3,
                         "ada_n_estimators": 300,
                         "ada_learning_rate": 0.2
                         },
        "RUSBoost"    : {"apply_smote": True,
                         "smote_strategy": 0.2,
                         "threshold": 0.3,
                         "rus_n_estimators": 300, 
                         "rus_learning_rate": 0.2,
                         },
        "Logit"       : {"apply_smote": True,
                         "smote_strategy": 0.2,
                         "threshold": 0.5,
                         "logit_C": 0.01, 
                         "logit_penalty": "l1",
                         },
        "SVM"         : {"apply_smote": True,
                         "smote_strategy": 0.2,
                         "threshold": 0.7,
                         "svm_C": 0.5, 
                         "svm_kernel": "rbf",
                         "svm_gamme": "scale"
                         },
        "MLP"         : {"apply_smote": True,
                         "smote_strategy": 0.2,
                         "threshold": 0.3,
                         "mlp_hidden_layer_sizes": (64, 32), 
                         "mlp_alpha": 0.0001,
                         "mlp_learning_rate_init": 0.01,
                         "mlp_activation": "relu"
                         },
    },
    # parent_dir=None  -> defaults to figures/sensitivity/best_configs/
    # parent_dir=C.DATA_PATH_OUTPUTS_FIG / "sensitivity" / "manual_configs" # for a test run
    run_shap=True,
)

# Side-by-side metrics of the per-model configurations
specific_models["df_report"]